**Part 1: What is an LLM?**

- A large language model is a decoder-only transformer (Day 20: stacked blocks with a causal mask) trained on a huge amount of text to do one thing: predict the next token.

- The chain: text -> tokens -> ids -> embedding + position -> N x [attention, feed-forward]
     -> the vector at the LAST position -> Linear (d -> vocabulary size) -> softmax
     -> a probability for every possible next token

**Generation is a loop**

The model never writes a whole answer at once. It does this:

1. Take the tokens so far: The capital of France is
2. Run the transformer; get probabilities for the next token, for example Paris 0.92, the 0.02, located 0.01, and so on, over the whole vocabulary (about 32,000 to 200,000 tokens).
3. Choose one token (Part 4 shows how: greedy, temperature, top-p).
4. Append it to the text and go back to step 1.

That is “autoregressive” generation. A 200-word answer is about 270 passes through the model.

**Why the causal mask makes training cheap**

- During training, one text of 1,000 tokens gives 1,000 training examples at once: position 1 predicts token 2, position 2 predicts token 3, and so on. The causal mask makes sure each position sees only the tokens on its left, so no position can cheat. This is why one pass over trillions of tokens of text is feasible.

Parameters: all the learned weights

- “A 7B model” has 7 billion parameters. You already know how to count them. For a decoder with model size d, N blocks, and vocabulary V (feed-forward width 4d):
    * per block ≈ 4 d²  (attention: W_Q, W_K, W_V, W_O)  +  8 d²  (feed-forward: d->4d->d)  =  12 d²
    * total    ≈ 12 · N · d²   +   V · d(the word-embedding table)

Check with the numbers you already have: BERT-base (d = 768) has 12 × 768² = 7.08 million per block. That is your 7,087,872 from EX 5, to within the small biases and norms.

A 7B-class model: d = 4096, N = 32, V = 32,000:

- blocks: 12 × 32 × 4096² = 6.44 billion
- embeddings: 32,000 × 4096 = 0.13 billion
- total ≈ 6.6 billion

- Memory: every parameter takes bytes: 4 bytes in float32, 2 in float16, about 0.5 in 4-bit. So 6.6 billion parameters in float16 is about 13 GB, and in 4-bit about 3.3 GB. That is why GPUs with 24 GB can run a 7B model, why quantisation exists, and why LoRA (Day 26) trains only a small part.

Your turn (10 minutes, paper)

E1. A model has d = 1024, N = 24 blocks and V = 50,000.

(a) The parameters in the blocks (12 · N · d²).
   * 12 x 24 x 1024^2 = 301,989,888

(b) The parameters in the embedding table (V · d).
   * 50,000 x 1024 = 51,200,000

(c) The total, and its memory in float16 (2 bytes per parameter), in MB or GB.
   * total: 301,989,888 + 51,200,000 = 353,189,888
   * memory in float16: 353,198,888 x 2 = 706,379,776 (706Mb)

E2. In your own words, 2 sentences: why can a decoder-only model train on all 1,000 positions of a text at once, and what stops it from cheating?
   * A decoder-only model can train on all 1,000 positions of a text at once because it has stacked blocks with causal mask to predict the next token, and it allows only to see the token to its left, that is what it stops from cheating. 
   * “all at once”: every position is its own training example (position 1 predicts token 2, position 2 predicts token 3, and so on), so one pass gives 1,000 predictions.

E3. Generation is a loop. A reply of 300 tokens needs how many passes through the model? And why does that explain why long answers take longer to appear?
   * A reply of 300 tokens needs 300 passes through the model. One forward pass for each generated token.
   * Because it must run through the whole forward pass for each token. Each pass needs the token chosen in the previous one, so the passes cannot be done in parallel, and time grows with the length of the answer.

**Part 2: Tokens and Context Window**

**Tokens: where the vocabulary comes from**

On Day 19 you split on spaces. LLMs use subword tokens, built by byte-pair encoding (BPE):

1. Start with single characters (or bytes) as the vocabulary.
2. Count every pair of neighbouring symbols in the training text (weighted by word frequency).
3. Merge the most frequent pair into one new symbol. Add it to the vocabulary.
4. Repeat until the vocabulary has the size you want (32,000 to 200,000).

Common words end up as one token, and rare words are split into known pieces, so there is no unknown word. Rules of thumb for English: 1 token is about 4 characters, or ¾ of a word. Other languages and code usually need more tokens per word.

**The context window**

- The model can only read a fixed number of tokens at once, the context window (for example 8,000 or 200,000 tokens). The prompt and the answer share it: system instructions + your question + any documents + the reply, all counted. Attention costs grow with the square of the length (Day 20), which is why windows are limited, and why Day 23 (RAG) retrieves only the relevant chunks instead of pasting everything.

APIs also charge per token, with separate prices for input and output tokens.

E4. BPE by hand. The training text has these words with counts (a character is separated from the next):

|word|count|               
|---|---|                    
|h u g|10|                 
|p u g|5|                  
|p u n|12|                 
|b u n|4|                  
|h u g s|5|  

(a) List all neighbouring pairs and their counts (hint: u g appears in hug, pug and hugs).

|Pair|count|              
|---|---|    
|'u','g'|20|   
|'p','u'|17|
|'h','u'|15|            
|'g','s'|5|
|'u','n'|16|
|'b','u'|4|

(b) Merge the most frequent pair. Rewrite the table with the new symbol.
merge 1
|Pair|count|              
|---|---|    
|h ug|15|   
|p ug|5|            
|h ug s|5|
|p u n|12|
|b u n|4|

(c) Count the pairs again and do the second merge, then the third.

merge 2
|Pair|count|              
|---|---|    
|'h','ug'|15|   
|'p','ug'|5|            
|'ug','s'|5|
|'u','n'|16|
|'b','u'|4|
|'p','u'|12|

after winner (u, n)
|Pair|count|              
|---|---|    
|h ug|10|   
|p ug|5|
|h ug s|5|
|b un|4|
|p un|12|

merge 3 
|Pair|count|              
|---|---|    
|hug|10|   
|p ug|5|            
|p un|12|
|b un|4|
|hug s|5|

(d) Starting vocabulary is the 7 letters b, g, h, n, p, s, u. How big is it after 3 merges?
   * 7 letters + 3 merges = 10 symbols: b, g, h, n, p, s, u, ug, un, hug.

E5. Context budget. A window of 8,000 tokens. The system prompt uses 500 tokens, the user’s question 300, and you reserve 1,000 for the answer. How many tokens are left for documents, and about how many English words is that (¾ of a word per token)?
* 500(system) + 300(user's question) + 1000(reserved) = 1,800 tokens
* tokens left: 8,000 - 1800 = 6,200 tokens
* 6,200 x 3/4 = 4,650 tokens

E6. Cost. With illustrative prices of $3 per million input tokens and $15 per million output tokens (not a real price list), one call uses 2,000 input tokens and 500 output tokens.
(a) The cost of one call.
   * Input cost: 2,000/1,000,000 x 3.00 = 0.006 
   * Output cost: 500/1,000,000 x 15.00 = 0.0075
   * Total per call: 0.006 + 0.0075 = 0.0135
(b) The cost of 10,000 calls.
   * 0.0135 x 10,000 = 135.00

**Part 3: Pretraining and instruction tuning**

*Stage 1: Pretraining (learn language)*

- Data: trillions of tokens of text (web pages, books, code).
- Task: at every position, predict the next token. The same causal-mask trick as Part 1.
- Loss: cross-entropy, the same family as the log-loss you used since Day 9:

loss at one position = - ln( p )      where p = probability the model gave to the TRUE next token

- If the model gave the true token 0.92, loss = -ln(0.92) = 0.083. If it gave 0.01, loss = 4.6. Confident and right costs almost nothing; confident and wrong costs a lot. The final loss is the average over all positions.

- Perplexity = e^(average loss). It reads as “the model is as unsure as if it were choosing among this many equally likely tokens”. Loss 1.03 gives perplexity 2.8.
- The result is a base model: it continues text well, but it does not follow instructions. Given What is the capital of France? it may continue with more quiz questions, because that is what often follows a question on the web.
- The cost is enormous: thousands of GPUs for weeks. You will not do this; you will use or fine-tune one (Day 26).

*Stage 2: Instruction tuning (learn to be an assistant)*

- Data: tens of thousands of examples of (instruction, ideal response), written by people, often in a chat format with roles (system, user, assistant).
- Task: the same next-token loss, but computed only on the response tokens, not on the instruction.
- Result: the model now answers the question instead of continuing it.

*Stage 3: Preference tuning (RLHF, DPO, and similar)*

- People compare two answers and mark the better one. The model is adjusted to produce answers like the preferred ones: more helpful, more honest, safer, better formatted.

Two limits that matter for the rest of the course:

   * Hallucination: the model produces plausible text, not facts looked up. If it did not see the answer in training, it may still produce a confident one.
   * Knowledge cutoff: it knows nothing after its training data. Both problems lead to RAG (Day 23): put the right documents into the prompt, so the model answers from them.

Your turn (10 minutes, paper)

E7. The true next tokens at three positions got these probabilities from the model: 0.5, 0.1, 0.9.

(a) The loss at each position, -ln(p). (Calculator: ln 0.5 = -0.693, ln 0.1 = -2.303, ln 0.9 = -0.105.)
   * p=0.5, -ln(0.5) = -(-0.693) = 0.693
   * p=0.5, -ln(0.1) = -(-2.303) = 2.303
   * p=0.5, -ln(0.9) = -(-0.105) = 0.105
(b) The average loss.
   * (0.693 + 2.303 + 0.105) / 3 = 1.034
(c) The perplexity, e^(average loss).
   * e^(1.034) = 2.81

E8. In instruction tuning the loss is computed only on the response tokens. Why not on the instruction tokens too? One or two sentences.
   * Instruction tokens are provided as fixed context by the user, so training the model to predict user inputs wastes capacity and encourages the model to generate prompts rather than answers. Computing loss only on response tokens forces the model to focus strictly on learning how to follow instructions and generate helpful completions.

E9. The prompt is Translate to French: cheese. Which kind of model, base or instruction-tuned, is more likely to answer fromage, and what might the other one write? Why?
   * Instruction-tuned model because it is specifically build for it, the other one (base) will are simple text-completers trained purely to predict the statistically most likely next word in a document format.

**Part 4: Inference, temperature and top-p**

- Inference means using a trained model to generate text. The model gives you probabilities for the next token, and you choose how to pick from them.

- From scores to probabilities, with temperature

- The last layer produces a logit (a raw score) for every token. You already know softmax from Day 19. Temperature T divides the logits first:
   * p_i = e^(z_i / T) / sum_j e^(z_j / T)

Using your Day 19 example, logits [2, 1, 0]:

|T|the three probabilities|effect
|---|---|---|
|0.5|0.867, 0.117, 0.016|sharper: the best token dominates
|1|0.665, 0.245, 0.090|the plain softmax you already know
|2|0.507, 0.307, 0.186|flatter: more even, more surprising choices

- T → 0 is greedy decoding: always pick the largest probability, so the same prompt gives the same answer.
- Higher T gives more variety and more risk of nonsense.

**Top-p (nucleus sampling)**

- Sort the tokens by probability. Keep the smallest set whose probabilities add up to at least p. Throw the rest away, renormalise, and sample from what is left. With p = 0.9 and probabilities [0.665, 0.245, 0.090]: the running sum is 0.665, then 0.910 which reaches 0.9, so keep the first two and renormalise: 0.665 / 0.910 = 0.731, 0.245 / 0.910 = 0.269. The long tail of unlikely tokens is cut off, so the rare weird choice can’t happen. Top-k keeps a fixed number of tokens instead of a fixed total probability.

The usual practice: change one of temperature or top-p, not both.

A few more inference words:

- Time to first token (reading the prompt, which can be done in parallel) versus tokens per second (generating, one pass per token, as in E3).
- The KV cache: the keys and values of the earlier tokens are stored, so each new token does not recompute them. It speeds generation but uses memory that grows with the context length.

Your turn (10 minutes, paper)

- E10. Logits [2, 1, 0]. Compute the probabilities at T = 0.5 and at T = 2. (Divide the logits by T first, then softmax. Use e^4 = 54.6, e^2 = 7.389, e^1 = 2.718, e^0.5 = 1.649.) Compare with the table above and check that each set adds up to 1.
   * for T = 0.5: 2/0.5, 1/0.5, 0/0.5 = [4, 2, 0]
   * find the exp of [4, 2, 0] and the total of it. [54.6, 7.389, 1.0], (62.989)
   * exp / total exp = [0.8668, 0.1173, 0.0159]
   * total = 1.0

   * for T = 2, [1, 0.5, 0]
   * find the exp of [4, 2, 0] and the total of it. [2.718, 1.649, 1], (5.367)
   * exp / total exp = [0.5064, 0.3072, 0.1863]
   * total = 1.0

- E11. Top-p with p = 0.8 on the T = 1 probabilities [0.665, 0.245, 0.090]. Which tokens are kept, and what are the renormalised probabilities?
   * Token 1 (0.665) < 0.8, Token 2 (0.665 + 0.245) >= 0.8 (threshold met)
   * Token 1 and 2 kept
   * Sum of kept probabilities: 0.665 + 0.245 = 0.910
   * Renormalized Token 1: 0.665/0.910 = 0.7308
   * Renormalized Token 2: 0.245/0.910 = 0.2692
   
- E12. For each task, choose a temperature (low, medium or high) and say why in one sentence: (a) extracting an invoice number from a document into JSON; (b) brainstorming 10 slogans for a new coffee shop.
   * For extracting an invoice number from a document into JSON need low temparature because data extraction requires strict factual precision, deterministic adherence to schema, and zero creative variance to avoid halluciating numbers.
   * For brainstorming 10 slogans for a new coffee shop needs high temparature because reative tasks would have produce many probability, so it would have many combinations instead of repeatetive answer.

**Part 5: Prompt engineering and calling an API**

1. Messages and roles

An LLM API is called with a list of messages, each with a role:

   * system: the rules for the whole conversation (who you are, format, limits). Written by the developer.
   * user: what the person says.
   * assistant: what the model said earlier.

*The model has no memory between calls. To continue a conversation you send the whole history again, so the input tokens (and the cost) grow with every turn.*

2. A good prompt has parts

Role and context → the task → constraints → the exact output format → examples. Be specific (“reply in JSON with keys invoice_no, total”) rather than vague (“extract the data”). Say what to do, and give the model the information it needs, because it cannot see what you can see.

3. Few-shot examples

Put 2 or 3 examples of input and the ideal output into the prompt. The model copies the pattern (format, tone, level of detail). This is often more effective than a longer description.

4. Structured output (JSON)

Ask for JSON with fixed keys, then parse and validate it in your own code (json.loads, check the keys and types). Never trust that it parsed. Plan for failure: if it fails, retry or ask the model to repair it. This is how an LLM becomes a component of a program (and what you will need for the RAG chatbot).

5. Tool (function) calling

The model cannot run code or look things up. With tool calling, you describe your functions (name, description, and a JSON schema of the parameters), and the model can answer with a request to call a tool instead of text. The loop:

1. You send the question and the tool descriptions.
2. The model replies: “call get_exchange_rate with {from: "MYR", to: "SGD"}”.
3. Your code runs the function (the model never does).
4. You send the result back as a new message.
5. The model writes the final answer using the result.

Your turn (15 minutes, paper)

- E13. Write a system prompt and one few-shot example (an input and the ideal output) for the invoice task. The JSON must have the keys invoice_no, total, currency and due_date. Use your own invoice text.
   * You are an invoice information extraction assistant.
Extract the invoice number, total amount, currency, and
due date from the provided invoice text.

   *   messages = [
         {"role": "user", "content": "ABC Technologies Sdn Bhd\nInvoice No: INV-2026-105\nInvoice Date: 2026-10-01\nDue Date: 2026-10-31\nTotal Amount: MYR 2,500.00"},
         {"role": "assistant", "content": '{"invoice_no": "INV-2026-105", "total": 2500.0, "currency": "MYR", "due_date": "2026-10-31"}'},
         {"role": "user", "content": new_invoice_text},   # the real one
      ]

- E14. The user asks, “How much is 350 MYR in SGD?” and you have given the model a tool get_exchange_rate(from, to). List the 5 steps of the loop in order. For each, say who acts, the model or your code.
   *  ABC Technologies Sdn Bhd
   *  Invoice No: INV-2026-105
   *  Invoice Date: 2026-10-01
   *  Due Date: 2026-10-31
   *  Total Amount: MYR 2,500.00
- E15. Your code calls json.loads(reply) and it fails, because the model wrapped the JSON in a markdown fence (three backticks, then json). Give two ways to make this robust.

   *   {
         "invoice_no": "INV-2026-105",
         "total": 2500.00,
         "currency": "MYR",
         "due_date": "2026-10-31"
      }

**Day 21 notes (for your notebook, rewrite in your words)**

|Q|Answer|
|---|---|
|Q1 How does an LLM produce text?|It predicts the next token from all earlier tokens, appends it, and |repeats (autoregressive).
|Q2 Why is it called a decoder-only model?|It uses only causal-masked attention blocks, so each token |sees only earlier tokens.
|Q3 What does temperature do?|It divides the logits by T before softmax. Low T is sharp and |near-greedy. High T is flatter and more random.
|Q4 Why do LLMs hallucinate, and what helps?|They are trained to produce likely text, not true text, |and they have a knowledge cutoff. RAG (Day 23) supplies real documents.
|Q5 Why is the API called stateless?|The model remembers nothing between calls. You resend the whole |message history each time, and it counts against the context window and the bill.
|Q6 How do you get reliable JSON?|System prompt with exact keys, one or two few-shot examples, then |parse and validate in code, with a retry.
|Q7 Who runs a tool?|Your code does. The model only requests the call, and your code returns the |result.